## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: PREPARACIÓN DE AMBIENTE (UNITY CATALOG)

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
-- 1. CONFIGURACIÓN DE PARÁMETROS (WIDGETS)
CREATE WIDGET TEXT storage_account DEFAULT 'adlspqrsdev';

In [0]:
%sql
-- 2. CREACIÓN DE UBICACIONES EXTERNAS (EXTERNAL LOCATIONS)
-- Usamos la credencial genérica que ya existe en el entorno del profesor
-- NOTA: ubicacion_metastore se omite; el contenedor 'metastore' ya está cubierto
-- por metastore_root_location registrada automáticamente por Unity Catalog.

CREATE EXTERNAL LOCATION IF NOT EXISTS ubicacion_raw
  URL 'abfss://raw-data@${storage_account}.dfs.core.windows.net/'
  WITH (STORAGE CREDENTIAL credential)
  COMMENT 'Ubicación externa para la ingesta de archivos crudos (JSON/CSV)';



In [0]:
%sql
-- 3. CREACIÓN DEL CATÁLOGO
CREATE CATALOG IF NOT EXISTS catalogo_pqrs
  COMMENT 'Catálogo principal - Proyecto PQRS';

-- Activamos el catálogo para crear los esquemas en su interior
USE CATALOG catalogo_pqrs;

In [0]:
%sql
-- Ubicación para la ingesta de archivos (Raw Data)
CREATE EXTERNAL LOCATION IF NOT EXISTS ubicacion_raw
  URL 'abfss://raw-data@${storage_account}.dfs.core.windows.net/'
  WITH (STORAGE CREDENTIAL credential)
  COMMENT 'Ubicación externa para la ingesta de archivos crudos (JSON/CSV)';

In [0]:
%sql
-- 4. CREACIÓN DE LAS CAPAS MEDALLÓN (SCHEMAS)
CREATE SCHEMA IF NOT EXISTS raw
  COMMENT 'Capa Raw: Registro de volúmenes externos y archivos crudos';

CREATE SCHEMA IF NOT EXISTS bronze
  COMMENT 'Capa Bronze: Datos crudos ingestados y tabulados directamente de la fuente';

CREATE SCHEMA IF NOT EXISTS silver
  COMMENT 'Capa Silver: Datos limpios, estandarizados y enriquecidos con IA';

CREATE SCHEMA IF NOT EXISTS gold
  COMMENT 'Capa Gold: Modelos de datos y métricas agregadas para Power BI';

In [0]:
%sql
-- 5. CREACIÓN DE TABLAS - CAPA BRONZE
CREATE TABLE IF NOT EXISTS bronze.tickets_raw (
    agente_id BIGINT,
    cliente_id BIGINT,
    cuerpo_mensaje STRING,
    detalles_tecnicos ARRAY<STRING>,
    ticket_id STRING,
    timestamp_apertura TIMESTAMP,
    ingestion_date TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.appendOnly' = 'true')
COMMENT 'Tabla inmutable con los datos crudos ingestados';

CREATE TABLE IF NOT EXISTS bronze.agentes_soporte_raw (
    agente_id BIGINT,
    agente_asignado STRING,
    region_agente STRING
) USING DELTA
COMMENT 'Catálogo crudo de agentes de soporte';

In [0]:
%sql
-- 6. CREACIÓN DE TABLAS - CAPA SILVER
CREATE TABLE IF NOT EXISTS silver.tickets_enriquecidos (
    ticket_id STRING,
    timestamp_apertura TIMESTAMP,
    agente_id BIGINT,
    agente_asignado STRING,
    region_agente STRING,
    canal_origen STRING,
    sistema_operativo STRING,
    tiempo_espera_segundos DOUBLE,
    cuerpo_mensaje STRING,
    criticidad INT,
    fecha_creacion DATE,
    nivel_soporte STRING,
    estado STRING,
    fecha_transformacion TIMESTAMP
) USING DELTA
COMMENT 'Tabla con datos aplanados, limpios y enriquecidos';

In [0]:
%sql
-- 7. CREACIÓN DE TABLAS Y DATAMARTS - CAPA GOLD

-- KPI Tendencias
CREATE TABLE IF NOT EXISTS gold.kpi_tendencias (
    fecha_apertura DATE,
    hora_apertura INT,
    tickets_recibidos BIGINT,
    espera_promedio_por_hora DOUBLE,
    fecha_actualizacion_kpi TIMESTAMP
) USING DELTA;

-- KPI Tecnología
CREATE TABLE IF NOT EXISTS gold.kpi_tecnologia (
    canal_origen STRING,
    sistema_operativo STRING,
    volumen_tickets BIGINT,
    impacto_critico BIGINT,
    fecha_actualizacion_kpi TIMESTAMP
) USING DELTA;

-- KPI Agentes
CREATE TABLE IF NOT EXISTS gold.kpi_agentes (
    agente_asignado STRING,
    region_agente STRING,
    total_tickets_atendidos BIGINT,
    total_tickets_criticos BIGINT,
    tiempo_promedio_espera_seg DOUBLE,
    fecha_actualizacion_kpi TIMESTAMP
) USING DELTA;